# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
# The IBUG cells later change directory (%cd ibug, %cd .., %cd ..) and then use
# "./content/drive/..." and "/content/drive/..." (Colab layout); on Kaggle both are linked
# to /kaggle/working so every Excel file lands in the same output folder.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d01_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # sediment_load train
    shutil.copy(glob.glob("/kaggle/input/**/d01_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # sediment_load test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: sediment_load data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: sediment_load data copied to ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795379359.605620      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795379379.292810      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (sediment_load_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['Qt', 'Qt-1', 'St-1']
TARGET_LABEL = 'Sediment load (St)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [7]:
train_data_path = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (1095, 4)
First 5 rows of training data:
       Qt   Qt-1    St-1      St
0  12.40  11.80  5930.0  5630.0
1   7.93  12.40  5630.0  1960.0
2   5.66   7.93  1960.0   491.0
3   5.04   5.66   491.0   692.0
4   4.84   5.04   692.0   452.0

Shape of test data: (365, 4)
First 5 rows of test data:
      Qt  Qt-1  St-1    St
0  5.27  5.49  45.0  44.0
1  5.13  5.27  44.0  29.0
2  4.84  5.13  29.0  23.0
3  5.04  4.84  23.0  26.0
4  5.21  5.04  26.0  30.0


In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (1095, 3)
Shape of y_train: (1095,)
Shape of X_test: (365, 3)
Shape of y_test: (365,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[-0.13088871 -0.15525695  1.01508124]
 [-0.31063021 -0.131129    0.95334642]
 [-0.40190834 -0.31088224  0.19812377]
 [-0.42683893 -0.40216633 -0.10417107]
 [-0.43488105 -0.42709855 -0.06280874]]

First five rows of normalized X_test:
[[-0.41759048 -0.40900259 -0.19595018]
 [-0.42321997 -0.4178495  -0.19615596]
 [-0.43488105 -0.42347936 -0.1992427 ]
 [-0.42683893 -0.4351412  -0.2004774 ]
 [-0.42000312 -0.42709855 -0.19986005]]


# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title(f'Predicted {TARGET_LABEL} vs. Actual {TARGET_LABEL}')
    plt.xlabel(f'Actual {TARGET_LABEL}')
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel(f'Predicted Mean {TARGET_LABEL}')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label=f'Actual {TARGET_LABEL}',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label=f'Predicted Mean {TARGET_LABEL}',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshjangra/sediment-load-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 276377.1029482182,
  'test_rmse': np.float64(525.7158005502765),
  'test_corr_coef': np.float64(0.9555378325000425),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jo

In [20]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.0},
  'test_mse': 276377.1029482182,
  'test_rmse': np.float64(525.7158005502765),
  'test_corr_coef': np.float64(0.9555378325000425),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 6162139.943307468,
  'cv_rmse': 2482.365795628732,
  'best_params': {'n_estimators': 500,
   'criterion': 'squared_error',
   'max_depth': 10,
   'min_samples_split': 5,
   'min_samples_leaf': 3,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': 100,
   'min_impurity_decrease': 0.1,
   'n_jo

# **Probabilistic distributions using NGBoost**

In [21]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize NGBoost model with best_params
ngb_model = NGBRegressor(Dist=Normal, **best_params)

# Fit the model
ngb_model.fit(X_train, y_train)

# Predict the distribution
pred_dist = ngb_model.pred_dist(X_test)

# Extract mean and standard deviation
predictions_NGB_df = pd.DataFrame({
    'Mean': pred_dist.loc,
    'StdDev': pred_dist.scale,
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, pred_dist.loc)
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_NGB_df.head())

Mean Squared Error: 612299.75
        Mean    StdDev  Actual
0  34.853193  8.435299    44.0
1  34.853193  6.883529    29.0
2  23.275575  5.788421    23.0
3  22.599803  6.242906    26.0
4  23.275575  6.494861    30.0


In [22]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


In [23]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,166.704185,8.767861,171.090633,329.075617,10


In [24]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,PIT_Value
0,0.860895
1,0.197574
2,0.481014
3,0.707003
4,0.849746
...,...
360,0.498537
361,0.410728
362,0.355070
363,0.309289


In [25]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.156164,0.056164
2,0.2,0.419178,0.219178
3,0.3,0.509589,0.209589
4,0.4,0.569863,0.169863
5,0.5,0.663014,0.163014
6,0.6,0.745205,0.145205
7,0.7,0.789041,0.089041
8,0.8,0.852055,0.052055
9,0.9,0.876712,0.023288


# **Probabilistic distributions using PGBM**

In [26]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
# Note: Adjust the incompatible keys based on PGBM's requirements
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params = best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Extract mean and standard deviation
# Note: PGBM might return predictions in a different format, adjust accordingly
predictions_PGBM_df = pd.DataFrame({
    'Mean': pred_dist.mean(axis=0),
    'StdDev': pred_dist.std(axis=0),
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, predictions_PGBM_df['Mean'])
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/500, Train metric: 4832.2310
Estimator 1/500, Train metric: 4807.3794
Estimator 2/500, Train metric: 4782.7339
Estimator 3/500, Train metric: 4758.2944
Estimator 4/500, Train metric: 4734.0581
Estimator 5/500, Train metric: 4710.0234
Estimator 6/500, Train metric: 4686.1880
Estimator 7/500, Train metric: 4662.5513
Estimator 8/500, Train metric: 4639.1113


Estimator 9/500, Train metric: 4615.8677
Estimator 10/500, Train metric: 4592.8184
Estimator 11/500, Train metric: 4569.9619
Estimator 12/500, Train metric: 4547.2974
Estimator 13/500, Train metric: 4524.8228
Estimator 14/500, Train metric: 4502.5371
Estimator 15/500, Train metric: 4480.4395
Estimator 16/500, Train metric: 4458.5283
Estimator 17/500, Train metric: 4436.8018
Estimator 18/500, Train metric: 4415.2593


Estimator 19/500, Train metric: 4393.8916
Estimator 20/500, Train metric: 4372.7129
Estimator 21/500, Train metric: 4351.7139
Estimator 22/500, Train metric: 4330.8936
Estimator 23/500, Train metric: 4310.2437
Estimator 24/500, Train metric: 4289.7783
Estimator 25/500, Train metric: 4269.4790
Estimator 26/500, Train metric: 4249.3628
Estimator 27/500, Train metric: 4229.4170
Estimator 28/500, Train metric: 4209.6416


Estimator 29/500, Train metric: 4190.0381
Estimator 30/500, Train metric: 4170.6011
Estimator 31/500, Train metric: 4151.3252
Estimator 32/500, Train metric: 4132.2241
Estimator 33/500, Train metric: 4113.2876
Estimator 34/500, Train metric: 4094.5076
Estimator 35/500, Train metric: 4075.8958
Estimator 36/500, Train metric: 4057.4482
Estimator 37/500, Train metric: 4039.1624
Estimator 38/500, Train metric: 4021.0266


Estimator 39/500, Train metric: 4003.0554
Estimator 40/500, Train metric: 3985.2427
Estimator 41/500, Train metric: 3967.5879
Estimator 42/500, Train metric: 3950.0837
Estimator 43/500, Train metric: 3932.7368
Estimator 44/500, Train metric: 3915.5391
Estimator 45/500, Train metric: 3898.4878
Estimator 46/500, Train metric: 3881.5945


... (58 outputs left out)

Estimator 485/500, Train metric: 2087.8928
Estimator 486/500, Train metric: 2087.6155
Estimator 487/500, Train metric: 2086.1196
Estimator 488/500, Train metric: 2085.8442
Estimator 489/500, Train metric: 2085.5781
Estimator 490/500, Train metric: 2084.0981
Estimator 491/500, Train metric: 2083.8257


Estimator 492/500, Train metric: 2083.5635
Estimator 493/500, Train metric: 2082.0869
Estimator 494/500, Train metric: 2081.8113
Estimator 495/500, Train metric: 2081.5459
Estimator 496/500, Train metric: 2081.2756
Estimator 497/500, Train metric: 2079.8174
Estimator 498/500, Train metric: 2078.3765


Estimator 499/500, Train metric: 2076.9460


Mean Squared Error: 194387.07
        Mean    StdDev  Actual
0  44.164482  1.480953    44.0
1  43.878284  1.706190    29.0
2  31.298843  1.480809    23.0
3  30.893499  1.541110    26.0
4  31.170538  1.383977    30.0


In [27]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


In [28]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,113.175372,54.398712,270.293808,329.075617,10


In [29]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,PIT_Value
0,4.557824e-01
1,1.388810e-18
2,1.045705e-08
3,7.483852e-04
4,1.988382e-01
...,...
360,5.653197e-08
361,1.374514e-10
362,1.034640e-06
363,1.879775e-06


In [30]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.786301,0.686301
2,0.2,0.802740,0.602740
3,0.3,0.810959,0.510959
4,0.4,0.813699,0.413699
5,0.5,0.824658,0.324658
6,0.6,0.827397,0.227397
7,0.7,0.835616,0.135616
8,0.8,0.838356,0.038356
9,0.9,0.838356,0.061644


# **Probabilistic Distribution using GPBoost (Treeffuser)**


In [31]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_score_engine = TreeBasedScoreModel(
    model_name='gpboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

# The fix is in the line below: '_score_model' has been corrected to 'score_model'.
model = Treeffuser(score_model=gpboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GPBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_TF_df.head())


[1]	valid_0's l2: 1.07944
[2]	valid_0's l2: 1.06677
[3]	valid_0's l2: 1.0542
[4]	valid_0's l2: 1.04202
[5]	valid_0's l2: 1.02993
[6]	valid_0's l2: 1.0184
[7]	valid_0's l2: 1.0068
[8]	valid_0's l2: 0.995717
[9]	valid_0's l2: 0.984777
[10]	valid_0's l2: 0.973933
[11]	valid_0's l2: 0.963153
[12]	valid_0's l2: 0.952649
[13]	valid_0's l2: 0.942242
[14]	valid_0's l2: 0.931951
[15]	valid_0's l2: 0.922031
[16]	valid_0's l2: 0.912107
[17]	valid_0's l2: 0.902208
[18]	valid_0's l2: 0.892626
[19]	valid_0's l2: 0.883093
[20]	valid_0's l2: 0.873824
[21]	valid_0's l2: 0.864496
[22]	valid_0's l2: 0.8558
[23]	valid_0's l2: 0.847352
[24]	valid_0's l2: 0.838929
[25]	valid_0's l2: 0.831118
[26]	valid_0's l2: 0.823128
[27]	valid_0's l2: 0.815197
[28]	valid_0's l2: 0.807453
[29]	valid_0's l2: 0.800339
[30]	valid_0's l2: 0.792804
[31]	valid_0's l2: 0.785845
[32]	valid_0's l2: 0.778689
[33]	valid_0's l2: 0.771375
[34]	valid_0's l2: 0.764457
[35]	valid_0's l2: 0.757497
[36]	valid_0's l2: 0.750871
[37]	valid_0'

[242]	valid_0's l2: 0.336195
[243]	valid_0's l2: 0.335476
[244]	valid_0's l2: 0.334986
[245]	valid_0's l2: 0.334444
[246]	valid_0's l2: 0.333961
[247]	valid_0's l2: 0.333517
[248]	valid_0's l2: 0.332919
[249]	valid_0's l2: 0.332464
[250]	valid_0's l2: 0.331947
[251]	valid_0's l2: 0.331262
[252]	valid_0's l2: 0.330732
[253]	valid_0's l2: 0.330248
[254]	valid_0's l2: 0.329821
[255]	valid_0's l2: 0.329223
[256]	valid_0's l2: 0.328739
[257]	valid_0's l2: 0.32836
[258]	valid_0's l2: 0.327703
[259]	valid_0's l2: 0.327154
[260]	valid_0's l2: 0.326642
[261]	valid_0's l2: 0.326212
[262]	valid_0's l2: 0.325915
[263]	valid_0's l2: 0.325414
[264]	valid_0's l2: 0.32503
[265]	valid_0's l2: 0.324651
[266]	valid_0's l2: 0.324138
[267]	valid_0's l2: 0.323695
[268]	valid_0's l2: 0.32331
[269]	valid_0's l2: 0.32303
[270]	valid_0's l2: 0.322747
[271]	valid_0's l2: 0.322247
[272]	valid_0's l2: 0.32188
[273]	valid_0's l2: 0.321259
[274]	valid_0's l2: 0.320742
[275]	valid_0's l2: 0.320492
[276]	valid_0's l2:

[462]	valid_0's l2: 0.26292
[463]	valid_0's l2: 0.262887
[464]	valid_0's l2: 0.262273
[465]	valid_0's l2: 0.261879
[466]	valid_0's l2: 0.261694
[467]	valid_0's l2: 0.261168
[468]	valid_0's l2: 0.260947
[469]	valid_0's l2: 0.260703
[470]	valid_0's l2: 0.260614
[471]	valid_0's l2: 0.260563
[472]	valid_0's l2: 0.260369
[473]	valid_0's l2: 0.260291
[474]	valid_0's l2: 0.260206
[475]	valid_0's l2: 0.260097
[476]	valid_0's l2: 0.259832
[477]	valid_0's l2: 0.259785
[478]	valid_0's l2: 0.259712
[479]	valid_0's l2: 0.259417
[480]	valid_0's l2: 0.259342
[481]	valid_0's l2: 0.259255
[482]	valid_0's l2: 0.259137
[483]	valid_0's l2: 0.259023
[484]	valid_0's l2: 0.258953
[485]	valid_0's l2: 0.258849
[486]	valid_0's l2: 0.25876
[487]	valid_0's l2: 0.258667
[488]	valid_0's l2: 0.258546
[489]	valid_0's l2: 0.258327
[490]	valid_0's l2: 0.257855
[491]	valid_0's l2: 0.257789
[492]	valid_0's l2: 0.257459
[493]	valid_0's l2: 0.256946
[494]	valid_0's l2: 0.256908
[495]	valid_0's l2: 0.256584
[496]	valid_0's 

          Mean       StdDev  Actual
0    87.533773  9358.793388    44.0
1   -20.323396  3400.262990    29.0
2  1314.882800  9403.015959    23.0
3   384.507253  3655.544044    26.0
4  -108.211818  6209.928732    30.0


In [32]:
generate_and_save_plots(predictions_GPBoost_TF_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


In [33]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,1686.928817,1230.019428,-128.16918,329.075617,10


In [34]:
generate_and_save_pit_results(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,PIT_Value
0,0.498144
1,0.505787
2,0.445361
3,0.460937
4,0.508878
...,...
360,0.473573
361,0.462848
362,0.512130
363,0.491556


In [35]:
generate_and_save_calibration_curve(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.002740,0.297260
4,0.4,0.076712,0.323288
5,0.5,0.761644,0.261644
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Gradient Boosting (Treeffuser)**

In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_score_engine = TreeBasedScoreModel(
    model_name='gradient_boosting',
    # FIX T1: the diffusion score model is fitted by least squares (Treeffuser's denoising
    # objective); the tuned GBR loss may be absolute_error/huber, so force squared_error here.
    model_params={**best_params, 'loss': 'squared_error'},
    seed=42,
)

model = Treeffuser(score_model=gbr_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GBR_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_TF_df.head())


         Mean      StdDev  Actual
0  167.912351  770.994271    44.0
1  216.725222  995.517408    29.0
2   92.365782  882.035495    23.0
3   66.762633  572.488530    26.0
4  135.825857  991.080178    30.0


In [37]:
generate_and_save_plots(predictions_GBR_TF_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


In [38]:
generate_and_save_crps_decomposition(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="Gradient Boosting",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,441.235453,310.271201,198.028822,329.075617,10


In [39]:
generate_and_save_pit_results(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,PIT_Value
0,0.436158
1,0.425215
2,0.468658
3,0.471618
4,0.457482
...,...
360,0.451405
361,0.442287
362,0.449706
363,0.449976


In [40]:
generate_and_save_calibration_curve(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.021918,0.278082
4,0.4,0.109589,0.290411
5,0.5,0.956164,0.456164
6,0.6,0.989041,0.389041
7,0.7,0.994521,0.294521
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Catboost (Treeffuser)**

In [41]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_score_engine = TreeBasedScoreModel(
    model_name='catboost',
    model_params=best_params,
    seed=42,
)

model = Treeffuser(score_model=catboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_CatBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_TF_df.head())

         Mean       StdDev  Actual
0   72.804612   564.083739    44.0
1 -224.419015  3946.194690    29.0
2   46.713724   600.719753    23.0
3  160.203157  1093.228423    26.0
4    5.162317   735.540828    30.0


In [42]:
generate_and_save_plots(predictions_CatBoost_TF_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


In [43]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,433.330606,263.46482,159.130962,329.075617,10


In [44]:
generate_and_save_pit_results(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,PIT_Value
0,0.479637
1,0.525602
2,0.484256
3,0.451149
4,0.513469
...,...
360,0.482045
361,0.518044
362,0.465305
363,0.557670


In [45]:
generate_and_save_calibration_curve(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.005479,0.194521
3,0.3,0.024658,0.275342
4,0.4,0.068493,0.331507
5,0.5,0.717808,0.217808
6,0.6,0.983562,0.383562
7,0.7,0.991781,0.291781
8,0.8,0.991781,0.191781
9,0.9,0.997260,0.097260


# **Probabilistic Distribution using XGBoost (Treeffuser)**

In [46]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_score_engine = TreeBasedScoreModel(
    model_name='xgboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=xgb_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_XGB_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_TF_df.head())

[0]	validation_0-rmse:0.99213


[1]	validation_0-rmse:0.94109


[2]	validation_0-rmse:0.89980


[3]	validation_0-rmse:0.89991


[4]	validation_0-rmse:0.90005


[5]	validation_0-rmse:0.86630


... (291 outputs left out)

[297]	validation_0-rmse:0.40941


[298]	validation_0-rmse:0.40946


[299]	validation_0-rmse:0.40869

         Mean       StdDev  Actual
0    8.505270  2392.742699    44.0
1   72.046352  1294.779586    29.0
2  217.665254  1606.401741    23.0
3  222.511714  1994.495063    26.0
4  -61.068753  3334.751892    30.0


In [47]:
generate_and_save_plots(predictions_XGB_TF_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


In [48]:
generate_and_save_crps_decomposition(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,553.209975,293.109651,68.880389,329.075617,10


In [49]:
generate_and_save_pit_results(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,PIT_Value
0,0.505918
1,0.486739
2,0.451774
3,0.460757
4,0.510893
...,...
360,0.476708
361,0.418792
362,0.430680
363,0.461380


In [50]:
generate_and_save_calibration_curve(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.010959,0.189041
3,0.3,0.049315,0.250685
4,0.4,0.084932,0.315068
5,0.5,0.838356,0.338356
6,0.6,0.991781,0.391781
7,0.7,0.997260,0.297260
8,0.8,0.997260,0.197260
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using LGBM (Treeffuser)**

In [51]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_score_engine = TreeBasedScoreModel(
    model_name='lightgbm',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=lgbm_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_LGBM_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_TF_df.head())


          Mean        StdDev  Actual
0  -430.774710  16243.529925    44.0
1  1192.576681  13844.753575    29.0
2  1551.881102  17294.729104    23.0
3 -1249.882653   9990.281077    26.0
4  -823.506107  13767.592535    30.0


In [52]:
generate_and_save_plots(predictions_LGBM_TF_df, y_test, "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


In [53]:
generate_and_save_crps_decomposition(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,3660.158739,2809.711655,-522.097673,329.075617,10


In [54]:
generate_and_save_pit_results(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,PIT_Value
0,0.511659
1,0.466510
2,0.464779
3,0.550812
4,0.524716
...,...
360,0.510580
361,0.490106
362,0.474331
363,0.495344


In [55]:
generate_and_save_calibration_curve(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.063014,0.336986
5,0.5,0.745205,0.245205
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **IBUG Installation**

In [56]:
# Clone the repository
!git clone https://github.com/jjbrophy47/ibug.git
%cd ibug

# Build extensions if needed
!python setup.py build_ext --inplace

# Install the package
!pip install .

# Verify installation
try:
    from ibug import IBUGWrapper
    print("IBUGWrapper imported successfully!")
except ImportError as e:
    print("Error importing IBUGWrapper:", e)

Cloning into 'ibug'...


remote: Enumerating objects: 1763, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.


remote: Total 1763 (delta 18), reused 15 (delta 4), pack-reused 1726 (from 1)
Receiving objects: 100% (1763/1763), 1.31 MiB | 13.66 MiB/s, done.


Resolving deltas: 100% (967/967), done.


/kaggle/working/ibug


Compiling ibug/parsers/_tree32.pyx because it changed.
Compiling ibug/parsers/_tree64.pyx because it changed.
[1/2] Cythonizing ibug/parsers/_tree32.pyx


... (16 outputs left out)

  Preparing metadata (pyproject.toml) ... done


  Created wheel for ibug: filename=ibug-0.0.12-cp313-cp313-linux_x86_64.whl size=1581579 sha256=01f218abf12eda85db2039350335ab592354f27b4e5e2c8c3c51b818d638e05b
  Stored in directory: /tmp/pip-ephem-wheel-cache-h4ujq_eg/wheels/48/c2/c1/d39bc91440bc7e7da40b146580d8aeb84c5361fa857fe8037e
Successfully built ibug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ibug]


IBUGWrapper imported successfully!


In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)  # FIX P2: seeded


# **Probabilistic Distributions using LGBM (IBUG)**

In [58]:
# y_train = y_train.to_numpy()
# y_val = y_val.to_numpy()

In [59]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# --- Ensure IBUG compatibility ---
best_params['reg_alpha'] = 0
best_params['class_weight'] = None
best_params['boosting_type'] = 'gbdt'

# Initialize and train the LightGBM regressor using the best_params
model = LGBMRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location2, scale2, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale**0.5
# Create a DataFrame for clarity
predictions_LGBM_df = pd.DataFrame({
    'Mean': location,
    'StdDev': dev,
    'Actual': y_test,
})

print(predictions_LGBM_df.head())

         Mean      StdDev  Actual
0  114.415059  475.848053    44.0
1  114.415059  475.848053    29.0
2  114.415059  475.848053    23.0
3  114.210966  475.848053    26.0
4  114.415059  475.848053    30.0


In [60]:
%cd ..

/kaggle/working


In [61]:
%cd ..

/kaggle


In [62]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx")

Excel file saved at ./content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


In [63]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx
   Sheets: CRPS_Summary_LGBM, CRPS_Bins_LGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LGBM,178.773472,88.537125,238.813469,329.075617,10


In [64]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,PIT_Value
0,0.441180
1,0.428772
2,0.423828
3,0.426467
4,0.429597
...,...
360,0.422351
361,0.420706
362,0.419885
363,0.419063


In [65]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.005479,0.094521
2,0.2,0.019178,0.180822
3,0.3,0.032877,0.267123
4,0.4,0.071233,0.328767
5,0.5,0.923288,0.423288
6,0.6,0.972603,0.372603
7,0.7,0.983562,0.283562
8,0.8,0.986301,0.186301
9,0.9,0.986301,0.086301


# **Probabilistic Distributions using XGBoost (IBUG)**

In [66]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

best_params['reg_alpha'] = 0
best_params['scale_pos_weight'] = 1
best_params['base_score'] = 0.5

# Initialize and train the XGBoost regressor using the best_params
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
# This returns the predicted mean (location) and uncertainty (scale, variance proxy)
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location, scale, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

# Calculate the standard deviation from the variance (scale)
std_dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale ** 0.5

# Create a DataFrame for clarity
predictions_XGB_df = pd.DataFrame({
    'Mean': location,
    'StdDev': std_dev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

        Mean      StdDev  Actual
0  40.724243  570.288563    44.0
1  40.724243  570.288563    29.0
2  36.132629  570.288563    23.0
3  34.585571  570.288502    26.0
4  34.585571  570.288502    30.0


In [67]:
generate_and_save_plots(predictions_XGB_df, y_test, "./content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx")  # FIX P3: was predictions_LGBM_df

Excel file saved at ./content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


In [68]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx
   Sheets: CRPS_Summary_XGB, CRPS_Bins_XGB


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGB,260.39753,75.871512,144.503702,329.075617,10


In [69]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,PIT_Value
0,0.502292
1,0.491799
2,0.490814
3,0.493994
4,0.496792
...,...
360,0.490497
361,0.489099
362,0.488399
363,0.487700


In [70]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.008219,0.091781
2,0.2,0.024658,0.175342
3,0.3,0.063014,0.236986
4,0.4,0.082192,0.317808
5,0.5,0.780822,0.280822
6,0.6,0.958904,0.358904
7,0.7,0.975342,0.275342
8,0.8,0.983562,0.183562
9,0.9,0.986301,0.086301


# **Matrix Evaulation**

In [71]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/sediment_load_uncertainty_analysis/probabilistic_distribution/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_TF_df, y_test, "LightGBM_TF")
    plot_metrics_and_save_to_excel(writer, predictions_XGB_TF_df, y_test, "XGBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_TF_df, y_test, "CatBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_TF_df, y_test, "GPBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_TF_df, y_test, "GradientBoosting_TF")